# ZeroBus + OpenTelemetry (demo notebook)

Boilerplate (Databricks Grafana secret merge, OTEL SDK → **Grafana Cloud OTLP/HTTP**, ZeroBus workspace, table, grants) is in **`zerobus_otel_lab.py`**. Fill Grafana OTLP JSON via **`secrets-bootstrap.ipynb`** or Databricks UI.

**ZeroBus Ingest OTel (preview / Beta):** Databricks also ships **native OpenTelemetry** for ZeroBus Ingest: OTLP **collector** endpoints over **HTTP** (and **gRPC** in some flows) for traces, logs, and metrics, so a standard OTel client can send data with **endpoint / exporter config** pointed at that integration. Telemetry is written into **Unity Catalog Delta** tables. **This notebook is different:** it points OTLP **only at Grafana Cloud** for dashboards and still uses **classic** ZeroBus ingest for table rows — it does **not** wire the SDK to ZeroBus Ingest OTEL’s Delta pipeline.

**UC + workspace OTLP (tables, headers, examples):** see **`docs/zerobus-ingest-otel-uc.md`**. **Workspace OTLP + ZeroBus demo:** **`notebooks/otel/uc/zerobus-uc-otel.ipynb`** (use a fresh kernel if you switch from this notebook).

**Grafana OTEL** Databricks secret: **`default_otel_config(otel_secret_scope=..., otel_secret_key=...)`** (defaults **`lfczerobusdemo`** / **`otel-grafana-rslee6392`**). **ZeroBus** JSON: **`default_zerobus_config(zerobus_secret_scope=..., zerobus_secret_key=...)`** (defaults match **`public_example.ipynb`**: **`lfczerobusdemo`** / **`lfczerobusdemo`**). Empty scope or key skips that secret’s load/save for that side.

**Auto-created service principal display name:** `zerobus-sp--<scope>--<secretKey>--ZEROBUS_OAUTH_SECRET` (secret location + JSON field for the OAuth client secret), unless **`ZEROBUS_SERVICE_PRINCIPAL_NAME`** is set — see **`zerobus_sp_display_name`** in **`zerobus_otel_lab.py`**.

The next cell imports the lab, merges the Grafana OTEL secret when scope/key are set, initializes telemetry, and bootstraps ZeroBus. Remaining cells: **ingest**, **verification**, **flush**.

**OTLP `401 Unauthorized`:** Grafana rejected `Authorization`. Re-copy **Connections → OpenTelemetry** credentials into the Databricks secret (or use **`secrets-bootstrap.ipynb`**). If **`GRAFANA_INSTANCE_ID`** and **`GRAFANA_API_TOKEN`** are both set, the lab builds Basic auth from them (preferred over a possibly stale **`GRAFANA_BASIC_AUTH_HEADER`**). Use a Cloud Access Policy token that can **push OTLP**; rotate the token if it was exposed.

**Direct Grafana link (like `Table:`):** set **`GRAFANA_STACK_URL`** (and optionally **`GRAFANA_TRACES_DATASOURCE_UID`**) in **`OTEL_CONFIG`** or the Grafana Databricks secret — see commented lines after **`merge_grafana_otel_secret`**. Then the notebook prints **`Grafana traces: https://…`** next to **`Table:`**.

**Traces vs metrics in Grafana:** Tempo **Table** (list) may show traces even when TraceQL looks empty; expand **`zerobus_ingestion`** — nested **`GET`** rows are normal (HTTP client instrumentation). **Metrics** from this notebook are **not** in Tempo: use **Explore → Metrics** with your **Prometheus / Mimir** datasource (**grafanacloud-*-prom**), e.g. PromQL `{__name__=~".*zerobus.*"}` after running the **flush** cell.


In [1]:
%pip install --quiet databricks-zerobus-ingest-sdk
%pip install --quiet opentelemetry-api opentelemetry-sdk opentelemetry-instrumentation
%pip install --quiet opentelemetry-exporter-otlp
%pip install --quiet opentelemetry-instrumentation-grpc opentelemetry-instrumentation-requests


Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
import sys
from pathlib import Path

_cwd = Path.cwd()
_lab_dir = None
for _root in [_cwd, *_cwd.parents]:
    if (_root / "notebooks" / "otel" / "grafana" / "zerobus_otel_lab.py").is_file():
        _lab_dir = _root / "notebooks" / "otel" / "grafana"
        break
if _lab_dir is None:
    raise RuntimeError("Could not find notebooks/otel/grafana/zerobus_otel_lab.py — use repo root as cwd.")
sys.path.insert(0, str(_lab_dir))

from zerobus_otel_lab import (
    bootstrap_zerobus_with_otel,
    create_otel_telemetry,
    default_otel_config,
    default_zerobus_config,
    flush_telemetry_and_save,
    merge_grafana_otel_secret,
)

# Databricks secrets (scope + key) — override per workspace.
_OTEL_SECRET_SCOPE = "lfczerobusdemo"
_OTEL_SECRET_KEY = "otel-grafana-rslee6392"
_ZEROBUS_SECRET_SCOPE = "lfczerobusdemo"
_ZEROBUS_SECRET_KEY = "lfczerobusdemo"

OTEL_CONFIG = default_otel_config(
    otel_secret_scope=_OTEL_SECRET_SCOPE,
    otel_secret_key=_OTEL_SECRET_KEY,
)
_config = default_zerobus_config(
    zerobus_secret_scope=_ZEROBUS_SECRET_SCOPE,
    zerobus_secret_key=_ZEROBUS_SECRET_KEY,
)
# Optional: _config["ZEROBUS_TABLE_NAME"] = "air_quality_otel"

merge_grafana_otel_secret(OTEL_CONFIG)
# Optional — same idea as "Table: …": prints "Grafana traces: https://…" after bootstrap / flush.
# Use your Grafana **browser** URL origin (not the OTLP gateway host). UID = Tempo datasource in Connections → Data sources.
# OTEL_CONFIG["GRAFANA_STACK_URL"] = "https://myorg.grafana.net"
# OTEL_CONFIG["GRAFANA_TRACES_DATASOURCE_UID"] = "grafanacloud-<stack>-traces"  # Settings → UID, not tempo-prod…/tempo URL

otel = create_otel_telemetry(OTEL_CONFIG, _config)
ctx = bootstrap_zerobus_with_otel(otel, _config, spark)

trace_operation = otel.trace_operation
record_metrics = otel.record_metrics
trace_provider = otel.trace_provider
meter_provider = otel.meter_provider
_w = ctx.w
username = ctx.username
_config = ctx.config
_config_original = ctx.config_original
fq_table_name = ctx.fq_table_name
table_catalog = ctx.table_catalog
table_schema = ctx.table_schema
table_name = ctx.table_name


def _save_config_if_changed() -> None:
    ctx.save_config_if_changed()


Loaded Grafana OTEL from scope='lfczerobusdemo' key='otel-grafana-rslee6392'
📊 OTEL: Grafana Authorization from GRAFANA_INSTANCE_ID + GRAFANA_API_TOKEN (Basic)
📊 OTEL: Grafana OTLP HTTP traces=https://otlp-gateway-prod-us-west-0.grafana.net/otlp/v1/traces metrics=https://otlp-gateway-prod-us-west-0.grafana.net/otlp/v1/metrics
✅ OpenTelemetry initialized successfully!
Loaded config from secret scope='lfczerobusdemo' key='lfczerobusdemo'
cfg['ZEROBUS_SERVER_ENDPOINT']='https://1444828305810485.zerobus.us-west-2.cloud.databricks.com'
cfg['DATABRICKS_WORKSPACE_URL']='https://e2-demo-field-eng.cloud.databricks.com'
SP exists: 'lfcdemo_zerobus_sp'  id=75332893425169
ZEROBUS_SERVICE_PRINCIPAL_ID = 75332893425169
ZEROBUS_APP_ID               = e6d2f259-8c72-4ac4-826b-f773af1528fc
client (app) secret is valid
table_catalog='main' table_schema='robert_lee' table_name='air_quality_otel'
USE CATALOG on main already granted
USE SCHEMA on main.robert_lee already granted
MODIFY on main.robert_lee.air

In [3]:
# INSTRUMENTED ZEROBUS INGESTION

import json
import time

from zerobus.sdk.shared import RecordType, StreamConfigurationOptions, TableProperties
from zerobus.sdk.sync import ZerobusSdk

with trace_operation(
    "zerobus_ingestion",
    {
        "table": fq_table_name,
        "endpoint": _config["ZEROBUS_SERVER_ENDPOINT"],
        "record_count": 10,
    },
) as main_span:
    with trace_operation("sdk_initialization"):
        sdk = ZerobusSdk(_config["ZEROBUS_SERVER_ENDPOINT"], _config["DATABRICKS_WORKSPACE_URL"])

    with trace_operation(
        "stream_creation", {"table": fq_table_name, "record_type": "JSON"}
    ) as stream_span:
        table_properties = TableProperties(fq_table_name)
        options = StreamConfigurationOptions(record_type=RecordType.JSON)
        stream = sdk.create_stream(
            _config["ZEROBUS_APP_ID"],
            _config["ZEROBUS_OAUTH_SECRET"],
            table_properties,
            options,
        )
        stream_span.set_attribute("stream_created", True)

    try:
        last_offset = None
        total_bytes = 0
        batch_start = time.time()

        for i in range(10):
            with trace_operation(
                f"ingest_record_{i}", {"record_index": i, "device_name": f"sensor-{i}"}
            ) as record_span:
                record_dict = {
                    "device_name": f"sensor-{i}",
                    "temp": 20 + i % 15,
                    "humidity": 50 + i % 40,
                }
                record_bytes = len(json.dumps(record_dict).encode("utf-8"))
                total_bytes += record_bytes
                record_span.set_attribute("record_bytes", record_bytes)
                last_offset = stream.ingest_record_offset(record_dict)
                record_span.set_attribute("offset", str(last_offset) if last_offset else "none")

        batch_duration_ms = (time.time() - batch_start) * 1000
        record_metrics("batch_ingest", records=10, bytes_size=total_bytes, duration_ms=batch_duration_ms)
        main_span.set_attribute("total_bytes", total_bytes)
        main_span.set_attribute("batch_duration_ms", batch_duration_ms)
        main_span.add_event(
            "Batch ingestion complete",
            {"records": 10, "bytes": total_bytes, "duration_ms": batch_duration_ms},
        )

        if last_offset is not None:
            with trace_operation("wait_for_commit", {"offset": str(last_offset)}) as wait_span:
                wait_start = time.time()
                stream.wait_for_offset(last_offset)
                wait_duration_ms = (time.time() - wait_start) * 1000
                wait_span.set_attribute("wait_duration_ms", wait_duration_ms)
                main_span.add_event(
                    "Offset committed",
                    {"offset": str(last_offset), "wait_duration_ms": wait_duration_ms},
                )

    finally:
        with trace_operation("stream_close"):
            stream.close()
            main_span.add_event("Stream closed")

print("✅ Ingestion complete — check your OTEL backend for traces.")
print(f"📊 Ingested 10 records, {total_bytes} bytes in {batch_duration_ms:.2f}ms")


2026-04-07T22:55:13.616768Z  INFO databricks_zerobus_ingest_sdk: Successfully created stream stream_id=f21c8405-39c3-4bd7-95c7-9ac38e712b63
2026-04-07T22:55:13.616948Z  INFO databricks_zerobus_ingest_sdk: Successfully created stream stream_id=f21c8405-39c3-4bd7-95c7-9ac38e712b63
2026-04-07T22:55:13.617054Z  INFO databricks_zerobus_ingest_sdk: Successfully created new ephemeral stream stream_id=f21c8405-39c3-4bd7-95c7-9ac38e712b63
2026-04-07T22:55:13.618719Z  INFO databricks_zerobus_ingest_sdk: Stream is not caught up to any offset yet. Waiting for the first offset. stream_id=f21c8405-39c3-4bd7-95c7-9ac38e712b63
2026-04-07T22:55:13.765289Z  INFO databricks_zerobus_ingest_sdk: Stream is caught up to the given offset. Waiting for acknowledgement completed. stream_id=f21c8405-39c3-4bd7-95c7-9ac38e712b63
2026-04-07T22:55:13.765764Z  INFO databricks_zerobus_ingest_sdk: Closing stream stream_id=f21c8405-39c3-4bd7-95c7-9ac38e712b63
2026-04-07T22:55:13.765773Z  INFO databricks_zerobus_ingest_sd

In [4]:
# Verify data

with trace_operation("data_verification") as span:
    result = spark.sql(
        f"select min(device_name), max(device_name), count(*) from {fq_table_name}"
    ).collect()[0]
    span.set_attribute("min_device", result[0])
    span.set_attribute("max_device", result[1])
    span.set_attribute("total_count", result[2])
    display(result)


Row(min(device_name)='sensor-0', max(device_name)='sensor-9', count(*)=40)

In [5]:
# Row count by device

with trace_operation("row_count_analysis") as span:
    result_df = spark.sql(
        f"""
    SELECT device_name, count(*) AS row_count
    FROM {fq_table_name}
    GROUP BY device_name
    HAVING count(*) > 1
    ORDER BY CAST(regexp_extract(device_name, '(\\\\d+)$', 1) AS INT), row_count DESC
    """
    )
    rows = result_df.collect()
    span.set_attribute("unique_devices", len(rows))
    span.set_attribute("max_duplicates", max((r["row_count"] for r in rows), default=0))
    display(result_df)


,device_name,row_count
0,sensor-0,5
1,sensor-1,5
2,sensor-2,5
3,sensor-3,5
4,sensor-4,5
5,sensor-5,5
6,sensor-6,5
7,sensor-7,5
8,sensor-8,5
9,sensor-9,5


In [6]:
# ZeroBus system table (if enabled)

with trace_operation("system_table_query") as span:
    try:
        result_df = spark.sql(
            f"""
        SELECT commit_time, table_name, committed_records, errors
        FROM system.lakeflow.zerobus_ingest
        WHERE table_name = '{fq_table_name}'
        ORDER BY commit_time DESC
        LIMIT 20
        """
        )
        rows = result_df.collect()
        span.set_attribute("system_table_available", True)
        span.set_attribute("commit_count", len(rows))
        if rows:
            total_records = sum(r["committed_records"] for r in rows if r["committed_records"])
            span.set_attribute("total_committed_records", total_records)
        display(result_df)
    except Exception as e:
        if "TABLE_OR_VIEW_NOT_FOUND" in str(e):
            span.set_attribute("system_table_available", False)
            span.add_event("System table not available")
            print("system.lakeflow.zerobus_ingest is not available in this workspace.")
        else:
            span.record_exception(e)
            raise


,commit_time,table_name,committed_records,errors
0,2026-04-07 22:37:20.751,main.robert_lee.air_quality_otel,10,[]


In [7]:
# Flush telemetry and save Zerobus config if changed

flush_telemetry_and_save(ctx)



📊 Flushing remaining telemetry...
Config unchanged — nothing to save

✅ OpenTelemetry instrumentation complete!
Grafana traces: https://rslee6392.grafana.net/explore?orgId=1&left=%7B%22datasource%22%3A%22grafanacloud-rslee6392-traces%22%2C%22queries%22%3A%5B%7B%22refId%22%3A%22A%22%2C%22datasource%22%3A%7B%22type%22%3A%22tempo%22%2C%22uid%22%3A%22grafanacloud-rslee6392-traces%22%7D%2C%22queryType%22%3A%22traceql%22%2C%22query%22%3A%22%7B%20resource.service.name%20%3D%20%5C%22zerobus-ingest%5C%22%20%7D%22%7D%5D%2C%22range%22%3A%7B%22from%22%3A%22now-1h%22%2C%22to%22%3A%22now%22%7D%7D
Grafana metrics: https://rslee6392.grafana.net/explore (open **Metrics**, search **zerobus_**; allow 1–3 min after export).
